# Customer Engagement & Product Utilization Analytics for Retention Strategy

## Notebook 04: Engagement vs Churn Analysis

### Notebook Objective

The objective of this notebook is to analyze how customer engagement affects churn and retention behavior.

This notebook uses the feature-engineered dataset created in Notebook 03. The focus is to understand whether active customers are more likely to stay with the bank compared with inactive customers.

---

### Key Business Questions

In this notebook, we will answer:

1. What is the churn rate of active vs inactive customers?
2. How much stronger is retention among active customers?
3. Which engagement segment contributes more to churn?
4. How does engagement interact with product depth?
5. How does engagement vary across geography?
6. Are high-value disengaged customers at higher churn risk?
7. How does relationship strength differ between active and inactive customers?

---

### Dashboard Relevance

This notebook supports the following dashboard module:

`Engagement vs Churn Overview`

The outputs from this notebook will help design KPIs, charts, filters, and insight cards for the Streamlit dashboard.

### Import Libraries

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.2f}".format)
print("Imported successfully")

Imported successfully


### Define Paths

In [2]:
# Define project paths
BASE_DIR = Path.cwd().parent

FEATURES_DATA_PATH = BASE_DIR / "data" / "processed" / "european_bank_features.csv"

print("Feature Dataset Path:", FEATURES_DATA_PATH)

Feature Dataset Path: C:\Users\Mohit\OneDrive\Documents\Data Analytics\Unified Mentor Projects\Customer Engagement & Product Utilization Analytics for Retention Strategy\data\processed\european_bank_features.csv


### Load Feature-Engineered Dataset

In [3]:
# Load feature-engineered dataset
df = pd.read_csv(FEATURES_DATA_PATH)

# Preview dataset
df.head()

,Year,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Churn_Status,Engagement_Status,Credit_Card_Status,Product_Depth_Segment,Balance_Segment,Salary_Segment,Age_Segment,Tenure_Segment,High_Value_Customer_Flag,High_Value_Customer_Status,High_Value_Disengaged_Flag,High_Value_Disengaged_Status,At_Risk_Premium_Flag,At_Risk_Premium_Status,Salary_Balance_Mismatch_Flag,Salary_Balance_Mismatch_Status,Customer_Risk_Segment,Relationship_Strength_Index,Relationship_Strength_Tier
0,2025,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,"101,348.88",1,Churned,Active Member,Has Credit Card,Single Product,Zero Balance,High Salary,Mid-Age Customer,New Customer,0,Standard Value Customer,0,Not High-Value Disengaged,0,Not At-Risk Premium,0,No Salary-Balance Mismatch,Standard Monitoring,55,Moderate Relationship
1,2025,15647311,Hill,608,Spain,Female,41,1,"83,807.86",1,0,1,"112,542.58",0,Retained,Active Member,No Credit Card,Single Product,Medium Balance,High Salary,Mid-Age Customer,New Customer,0,Standard Value Customer,0,Not High-Value Disengaged,0,Not At-Risk Premium,0,No Salary-Balance Mismatch,Standard Monitoring,55,Moderate Relationship
2,2025,15619304,Onio,502,France,Female,42,8,"159,660.80",3,1,0,"113,931.57",1,Churned,Inactive Member,Has Credit Card,High Product Load,Premium Balance,High Salary,Mid-Age Customer,Established Relationship,1,High Value Customer,1,High-Value Disengaged,1,At-Risk Premium Customer,0,No Salary-Balance Mismatch,At-Risk Premium,50,Weak Relationship
3,2025,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,"93,826.63",0,Retained,Inactive Member,No Credit Card,Optimal Product Depth,Zero Balance,Medium Salary,Mid-Age Customer,New Customer,0,Standard Value Customer,0,Not High-Value Disengaged,0,Not At-Risk Premium,0,No Salary-Balance Mismatch,Standard Monitoring,35,Weak Relationship
4,2025,15737888,Mitchell,850,Spain,Female,43,2,"125,510.82",1,1,1,"79,084.10",0,Retained,Active Member,Has Credit Card,Single Product,High Balance,Medium Salary,Mid-Age Customer,New Customer,0,Standard Value Customer,0,Not High-Value Disengaged,0,Not At-Risk Premium,0,No Salary-Balance Mismatch,Standard Monitoring,75,Moderate Relationship


### Dataset Shape Check

In [4]:
print(f"Total Rows: {df.shape[0]:,}")
print(f"Total Columns: {df.shape[1]:,}")

Total Rows: 10,000
Total Columns: 33


### Engagement Distribution

In [5]:
# Analyze active vs inactive customer distribution
engagement_distribution = (
    df["Engagement_Status"]
    .value_counts()
    .reset_index()
)

engagement_distribution.columns = ["Engagement Status", "Customer Count"]

engagement_distribution["Customer %"] = (
    engagement_distribution["Customer Count"] / len(df) * 100
).round(2)

engagement_distribution

,Engagement Status,Customer Count,Customer %
0,Active Member,5151,51.51
1,Inactive Member,4849,48.49


### Engagement vs Churn Summary

In [6]:
# Analyze churn and retention by engagement status
engagement_churn_summary = (
    df
    .groupby("Engagement_Status")
    .agg(
        Customer_Count=("CustomerId", "count"),
        Retained_Customers=("Exited", lambda x: (x == 0).sum()),
        Churned_Customers=("Exited", "sum"),
        Churn_Rate=("Exited", "mean")
    )
    .reset_index()
)

engagement_churn_summary["Retention_Rate"] = (
    100 - engagement_churn_summary["Churn_Rate"] * 100
).round(2)

engagement_churn_summary["Churn_Rate"] = (
    engagement_churn_summary["Churn_Rate"] * 100
).round(2)

engagement_churn_summary = engagement_churn_summary[
    [
        "Engagement_Status",
        "Customer_Count",
        "Retained_Customers",
        "Churned_Customers",
        "Retention_Rate",
        "Churn_Rate"
    ]
].sort_values(by="Churn_Rate", ascending=False)

engagement_churn_summary

,Engagement_Status,Customer_Count,Retained_Customers,Churned_Customers,Retention_Rate,Churn_Rate
1,Inactive Member,4849,3547,1302,73.15,26.85
0,Active Member,5151,4416,735,85.73,14.27


### Engagement Retention Ratio KPI

In [7]:
# Calculate retention and churn comparison between active and inactive members
active_retention_rate = engagement_churn_summary.loc[
    engagement_churn_summary["Engagement_Status"] == "Active Member",
    "Retention_Rate"
].iloc[0]

inactive_retention_rate = engagement_churn_summary.loc[
    engagement_churn_summary["Engagement_Status"] == "Inactive Member",
    "Retention_Rate"
].iloc[0]

active_churn_rate = engagement_churn_summary.loc[
    engagement_churn_summary["Engagement_Status"] == "Active Member",
    "Churn_Rate"
].iloc[0]

inactive_churn_rate = engagement_churn_summary.loc[
    engagement_churn_summary["Engagement_Status"] == "Inactive Member",
    "Churn_Rate"
].iloc[0]

engagement_retention_ratio = active_retention_rate / inactive_retention_rate
churn_risk_multiplier = inactive_churn_rate / active_churn_rate

print(f"Active Customer Retention Rate: {active_retention_rate:.2f}%")
print(f"Inactive Customer Retention Rate: {inactive_retention_rate:.2f}%")
print(f"Engagement Retention Ratio: {engagement_retention_ratio:.2f}x")
print(f"Inactive Customer Churn Risk Multiplier: {churn_risk_multiplier:.2f}x")

Active Customer Retention Rate: 85.73%
Inactive Customer Retention Rate: 73.15%
Engagement Retention Ratio: 1.17x
Inactive Customer Churn Risk Multiplier: 1.88x


### Churn Contribution by Engagement Status

In [8]:
# Analyze how much each engagement segment contributes to total churn
total_churned_customers = df["Exited"].sum()

engagement_churn_contribution = (
    df
    .groupby("Engagement_Status")
    .agg(
        Customer_Count=("CustomerId", "count"),
        Churned_Customers=("Exited", "sum")
    )
    .reset_index()
)

engagement_churn_contribution["Churn_Contribution_%"] = (
    engagement_churn_contribution["Churned_Customers"] / total_churned_customers * 100
).round(2)

engagement_churn_contribution["Customer_Share_%"] = (
    engagement_churn_contribution["Customer_Count"] / len(df) * 100
).round(2)

engagement_churn_contribution = engagement_churn_contribution.sort_values(
    by="Churn_Contribution_%",
    ascending=False
)

engagement_churn_contribution

,Engagement_Status,Customer_Count,Churned_Customers,Churn_Contribution_%,Customer_Share_%
1,Inactive Member,4849,1302,63.92,48.49
0,Active Member,5151,735,36.08,51.51


### Engagement by Product Depth

In [9]:
# Analyze engagement distribution by product depth segment
engagement_product_depth = (
    df
    .groupby(["Product_Depth_Segment", "Engagement_Status"])
    .agg(
        Customer_Count=("CustomerId", "count")
    )
    .reset_index()
)

engagement_product_depth["Segment_Total"] = (
    engagement_product_depth
    .groupby("Product_Depth_Segment")["Customer_Count"]
    .transform("sum")
)

engagement_product_depth["Customer_%"] = (
    engagement_product_depth["Customer_Count"] / engagement_product_depth["Segment_Total"] * 100
).round(2)

engagement_product_depth

,Product_Depth_Segment,Engagement_Status,Customer_Count,Segment_Total,Customer_%
0,High Product Load,Active Member,142,326,43.56
1,High Product Load,Inactive Member,184,326,56.44
2,Optimal Product Depth,Active Member,2446,4590,53.29
3,Optimal Product Depth,Inactive Member,2144,4590,46.71
4,Single Product,Active Member,2563,5084,50.41
5,Single Product,Inactive Member,2521,5084,49.59


### Product Depth and Engagement vs Churn

In [10]:
# Analyze churn by product depth and engagement status
product_engagement_churn = (
    df
    .groupby(["Product_Depth_Segment", "Engagement_Status"])
    .agg(
        Customer_Count=("CustomerId", "count"),
        Churned_Customers=("Exited", "sum"),
        Churn_Rate=("Exited", "mean"),
        Avg_Relationship_Strength=("Relationship_Strength_Index", "mean")
    )
    .reset_index()
)

product_engagement_churn["Churn_Rate"] = (
    product_engagement_churn["Churn_Rate"] * 100
).round(2)

product_engagement_churn["Avg_Relationship_Strength"] = (
    product_engagement_churn["Avg_Relationship_Strength"]
).round(2)

product_engagement_churn = product_engagement_churn.sort_values(
    by="Churn_Rate",
    ascending=False
)

product_engagement_churn

,Product_Depth_Segment,Engagement_Status,Customer_Count,Churned_Customers,Churn_Rate,Avg_Relationship_Strength
1,High Product Load,Inactive Member,184,166,90.22,36.09
0,High Product Load,Active Member,142,114,80.28,64.93
5,Single Product,Inactive Member,2521,924,36.65,43.42
4,Single Product,Active Member,2563,485,18.92,72.93
3,Optimal Product Depth,Inactive Member,2144,212,9.89,56.46
2,Optimal Product Depth,Active Member,2446,136,5.56,86.31


### Engagement by Balance Segment

In [11]:
# Analyze engagement distribution by balance segment
engagement_balance = (
    df
    .groupby(["Balance_Segment", "Engagement_Status"])
    .agg(
        Customer_Count=("CustomerId", "count")
    )
    .reset_index()
)

engagement_balance["Segment_Total"] = (
    engagement_balance
    .groupby("Balance_Segment")["Customer_Count"]
    .transform("sum")
)

engagement_balance["Customer_%"] = (
    engagement_balance["Customer_Count"] / engagement_balance["Segment_Total"] * 100
).round(2)

engagement_balance = engagement_balance.sort_values(
    by=["Balance_Segment", "Customer_%"],
    ascending=[True, False]
)

engagement_balance

,Balance_Segment,Engagement_Status,Customer_Count,Segment_Total,Customer_%
0,High Balance,Active Member,1958,3830,51.12
1,High Balance,Inactive Member,1872,3830,48.88
2,Low Balance,Active Member,40,75,53.33
3,Low Balance,Inactive Member,35,75,46.67
4,Medium Balance,Active Member,795,1509,52.68
5,Medium Balance,Inactive Member,714,1509,47.32
6,Premium Balance,Active Member,485,969,50.05
7,Premium Balance,Inactive Member,484,969,49.95
8,Zero Balance,Active Member,1873,3617,51.78
9,Zero Balance,Inactive Member,1744,3617,48.22


### Balance Segment and Engagement vs Churn

In [12]:
# Analyze churn by balance segment and engagement status
balance_engagement_churn = (
    df
    .groupby(["Balance_Segment", "Engagement_Status"])
    .agg(
        Customer_Count=("CustomerId", "count"),
        Churned_Customers=("Exited", "sum"),
        Churn_Rate=("Exited", "mean"),
        Avg_Balance=("Balance", "mean")
    )
    .reset_index()
)

balance_engagement_churn["Churn_Rate"] = (
    balance_engagement_churn["Churn_Rate"] * 100
).round(2)

balance_engagement_churn["Avg_Balance"] = (
    balance_engagement_churn["Avg_Balance"]
).round(2)

balance_engagement_churn = balance_engagement_churn.sort_values(
    by="Churn_Rate",
    ascending=False
)

balance_engagement_churn

,Balance_Segment,Engagement_Status,Customer_Count,Churned_Customers,Churn_Rate,Avg_Balance
3,Low Balance,Inactive Member,35,18,51.43,"38,454.02"
1,High Balance,Inactive Member,1872,627,33.49,"123,727.96"
7,Premium Balance,Inactive Member,484,145,29.96,"166,625.40"
5,Medium Balance,Inactive Member,714,192,26.89,"84,612.43"
2,Low Balance,Active Member,40,8,20.00,"40,316.85"
0,High Balance,Active Member,1958,360,18.39,"123,661.01"
9,Zero Balance,Inactive Member,1744,320,18.35,0.00
6,Premium Balance,Active Member,485,79,16.29,"166,909.65"
4,Medium Balance,Active Member,795,108,13.58,"83,197.73"
8,Zero Balance,Active Member,1873,180,9.61,0.00


### Engagement by Geography

In [13]:
# Analyze engagement distribution by geography
engagement_geography = (
    df
    .groupby(["Geography", "Engagement_Status"])
    .agg(
        Customer_Count=("CustomerId", "count")
    )
    .reset_index()
)

engagement_geography["Geography_Total"] = (
    engagement_geography
    .groupby("Geography")["Customer_Count"]
    .transform("sum")
)

engagement_geography["Customer_%"] = (
    engagement_geography["Customer_Count"] / engagement_geography["Geography_Total"] * 100
).round(2)

engagement_geography = engagement_geography.sort_values(
    by=["Geography", "Customer_%"],
    ascending=[True, False]
)

engagement_geography

,Geography,Engagement_Status,Customer_Count,Geography_Total,Customer_%
0,France,Active Member,2591,5014,51.68
1,France,Inactive Member,2423,5014,48.32
3,Germany,Inactive Member,1261,2509,50.26
2,Germany,Active Member,1248,2509,49.74
4,Spain,Active Member,1312,2477,52.97
5,Spain,Inactive Member,1165,2477,47.03


### Geography and Engagement vs Churn

In [14]:
# Analyze churn by geography and engagement status
geography_engagement_churn = (
    df
    .groupby(["Geography", "Engagement_Status"])
    .agg(
        Customer_Count=("CustomerId", "count"),
        Churned_Customers=("Exited", "sum"),
        Churn_Rate=("Exited", "mean"),
        Avg_Relationship_Strength=("Relationship_Strength_Index", "mean")
    )
    .reset_index()
)

geography_engagement_churn["Churn_Rate"] = (
    geography_engagement_churn["Churn_Rate"] * 100
).round(2)

geography_engagement_churn["Avg_Relationship_Strength"] = (
    geography_engagement_churn["Avg_Relationship_Strength"]
).round(2)

geography_engagement_churn = geography_engagement_churn.sort_values(
    by="Churn_Rate",
    ascending=False
)

geography_engagement_churn

,Geography,Engagement_Status,Customer_Count,Churned_Customers,Churn_Rate,Avg_Relationship_Strength
3,Germany,Inactive Member,1261,518,41.08,54.37
2,Germany,Active Member,1248,296,23.72,85.13
5,Spain,Inactive Member,1165,272,23.35,46.97
1,France,Inactive Member,2423,512,21.13,47.00
0,France,Active Member,2591,298,11.50,77.21
4,Spain,Active Member,1312,141,10.75,76.96


### High-Value Disengaged Customer Analysis

In [15]:
# Analyze high-value disengaged customer behavior
high_value_disengaged_analysis = (
    df
    .groupby("High_Value_Disengaged_Status")
    .agg(
        Customer_Count=("CustomerId", "count"),
        Churned_Customers=("Exited", "sum"),
        Churn_Rate=("Exited", "mean"),
        Avg_Balance=("Balance", "mean"),
        Avg_Salary=("EstimatedSalary", "mean"),
        Avg_Num_Products=("NumOfProducts", "mean"),
        Avg_Relationship_Strength=("Relationship_Strength_Index", "mean")
    )
    .reset_index()
)

high_value_disengaged_analysis["Churn_Rate"] = (
    high_value_disengaged_analysis["Churn_Rate"] * 100
).round(2)

for col in ["Avg_Balance", "Avg_Salary", "Avg_Num_Products", "Avg_Relationship_Strength"]:
    high_value_disengaged_analysis[col] = high_value_disengaged_analysis[col].round(2)

high_value_disengaged_analysis = high_value_disengaged_analysis.sort_values(
    by="Churn_Rate",
    ascending=False
)

high_value_disengaged_analysis

,High_Value_Disengaged_Status,Customer_Count,Churned_Customers,Churn_Rate,Avg_Balance,Avg_Salary,Avg_Num_Products,Avg_Relationship_Strength
0,High-Value Disengaged,2154,626,29.06,"108,144.37","132,331.13",1.47,51.63
1,Not High-Value Disengaged,7846,1411,17.98,"67,794.53","91,238.99",1.55,67.96


### Relationship Strength by Engagement

In [16]:
# Analyze relationship strength by engagement status
relationship_strength_engagement = (
    df
    .groupby("Engagement_Status")
    .agg(
        Customer_Count=("CustomerId", "count"),
        Avg_Relationship_Strength=("Relationship_Strength_Index", "mean"),
        Median_Relationship_Strength=("Relationship_Strength_Index", "median"),
        Min_Relationship_Strength=("Relationship_Strength_Index", "min"),
        Max_Relationship_Strength=("Relationship_Strength_Index", "max"),
        Churn_Rate=("Exited", "mean")
    )
    .reset_index()
)

relationship_strength_engagement["Avg_Relationship_Strength"] = (
    relationship_strength_engagement["Avg_Relationship_Strength"]
).round(2)

relationship_strength_engagement["Median_Relationship_Strength"] = (
    relationship_strength_engagement["Median_Relationship_Strength"]
).round(2)

relationship_strength_engagement["Churn_Rate"] = (
    relationship_strength_engagement["Churn_Rate"] * 100
).round(2)

relationship_strength_engagement = relationship_strength_engagement.sort_values(
    by="Avg_Relationship_Strength",
    ascending=False
)

relationship_strength_engagement

,Engagement_Status,Customer_Count,Avg_Relationship_Strength,Median_Relationship_Strength,Min_Relationship_Strength,Max_Relationship_Strength,Churn_Rate
0,Active Member,5151,79.07,80.00,40,110,14.27
1,Inactive Member,4849,48.91,50.00,10,80,26.85


### Engagement and Relationship Strength Tier Matrix

In [17]:
# Analyze relationship strength tiers by engagement status
engagement_tier_matrix = (
    df
    .groupby(["Engagement_Status", "Relationship_Strength_Tier"])
    .agg(
        Customer_Count=("CustomerId", "count"),
        Churned_Customers=("Exited", "sum"),
        Churn_Rate=("Exited", "mean")
    )
    .reset_index()
)

engagement_tier_matrix["Churn_Rate"] = (
    engagement_tier_matrix["Churn_Rate"] * 100
).round(2)

engagement_tier_matrix["Engagement_Total"] = (
    engagement_tier_matrix
    .groupby("Engagement_Status")["Customer_Count"]
    .transform("sum")
)

engagement_tier_matrix["Customer_%_Within_Engagement"] = (
    engagement_tier_matrix["Customer_Count"] / engagement_tier_matrix["Engagement_Total"] * 100
).round(2)

engagement_tier_matrix = engagement_tier_matrix.sort_values(
    by=["Engagement_Status", "Churn_Rate"],
    ascending=[True, False]
)

engagement_tier_matrix

,Engagement_Status,Relationship_Strength_Tier,Customer_Count,Churned_Customers,Churn_Rate,Engagement_Total,Customer_%_Within_Engagement
2,Active Member,Weak Relationship,91,42,46.15,5151,1.77
0,Active Member,Moderate Relationship,2267,391,17.25,5151,44.01
1,Active Member,Strong Relationship,2793,302,10.81,5151,54.22
5,Inactive Member,Very Weak Relationship,262,132,50.38,4849,5.40
6,Inactive Member,Weak Relationship,2778,812,29.23,4849,57.29
4,Inactive Member,Strong Relationship,80,16,20.00,4849,1.65
3,Inactive Member,Moderate Relationship,1729,342,19.78,4849,35.66


## Engagement Analysis Insight Summary

The engagement analysis confirms that customer activity is a major driver of retention behavior.

Key findings:

- Active customers show a much higher retention rate than inactive customers.
- Inactive customers have nearly twice the churn risk of active customers.
- Inactive customers contribute a larger share of total churn despite representing less than half of the customer base.
- Engagement becomes more meaningful when combined with product depth.
- Customers with optimal product depth and active status show the strongest retention behavior.
- High-value disengaged customers represent an important retention opportunity because they combine financial value with low activity.
- Engagement also improves the Relationship Strength Index, making it useful for dashboard scoring and customer prioritization.

These insights will directly support the Streamlit dashboard module: `Engagement vs Churn Overview`.

### Engagement KPI Summary Table

In [18]:
# Create dashboard-ready engagement KPI summary
engagement_kpi_summary = pd.DataFrame({
    "KPI": [
        "Active Customer Retention Rate",
        "Inactive Customer Retention Rate",
        "Engagement Retention Ratio",
        "Inactive Customer Churn Risk Multiplier",
        "Inactive Churn Contribution",
        "Active Churn Contribution"
    ],
    "Value": [
        f"{active_retention_rate:.2f}%",
        f"{inactive_retention_rate:.2f}%",
        f"{engagement_retention_ratio:.2f}x",
        f"{churn_risk_multiplier:.2f}x",
        f"{engagement_churn_contribution.loc[engagement_churn_contribution['Engagement_Status'] == 'Inactive Member', 'Churn_Contribution_%'].iloc[0]:.2f}%",
        f"{engagement_churn_contribution.loc[engagement_churn_contribution['Engagement_Status'] == 'Active Member', 'Churn_Contribution_%'].iloc[0]:.2f}%"
    ],
    "Business Meaning": [
        "Retention rate among active customers",
        "Retention rate among inactive customers",
        "How much stronger active customer retention is compared with inactive customers",
        "How much higher inactive churn risk is compared with active customers",
        "Share of total churn coming from inactive customers",
        "Share of total churn coming from active customers"
    ]
})

engagement_kpi_summary

,KPI,Value,Business Meaning
0,Active Customer Retention Rate,85.73%,Retention rate among active customers
1,Inactive Customer Retention Rate,73.15%,Retention rate among inactive customers
2,Engagement Retention Ratio,1.17x,How much stronger active customer retention is...
3,Inactive Customer Churn Risk Multiplier,1.88x,How much higher inactive churn risk is compare...
4,Inactive Churn Contribution,63.92%,Share of total churn coming from inactive cust...
5,Active Churn Contribution,36.08%,Share of total churn coming from active customers


### Dashboard-Ready Engagement Insight Table

In [19]:
# Create dashboard-ready insight table for engagement analysis
engagement_insight_table = pd.DataFrame({
    "Insight Area": [
        "Engagement Retention Gap",
        "Churn Contribution Risk",
        "Product Depth Risk",
        "Best Retention Profile",
        "Geographic Risk Segment",
        "High-Value Disengagement Risk",
        "Relationship Strength Signal"
    ],
    "Key Finding": [
        "Active customers retain better than inactive customers.",
        "Inactive customers contribute more churn than their customer share.",
        "High product load customers show extreme churn risk.",
        "Active customers with optimal product depth show the strongest retention.",
        "Germany inactive customers show the highest geography-engagement churn risk.",
        "High-value disengaged customers show elevated churn risk.",
        "Active customers have much stronger relationship strength than inactive customers."
    ],
    "Metric Evidence": [
        f"Active retention: {active_retention_rate:.2f}% vs inactive retention: {inactive_retention_rate:.2f}%",
        "Inactive customers: 48.49% customer share but 63.92% churn contribution",
        "High Product Load + Inactive churn rate: 90.22%",
        "Optimal Product Depth + Active churn rate: 5.56%",
        "Germany + Inactive churn rate: 41.08%",
        "High-value disengaged churn rate: 29.06%",
        "Active RSI: 79.07 vs inactive RSI: 48.91"
    ],
    "Dashboard Use": [
        "KPI card",
        "Churn contribution chart",
        "Product-depth risk chart",
        "Retention profile insight card",
        "Geography risk chart",
        "High-value disengaged detector",
        "Relationship Strength KPI"
    ]
})

engagement_insight_table

,Insight Area,Key Finding,Metric Evidence,Dashboard Use
0,Engagement Retention Gap,Active customers retain better than inactive c...,Active retention: 85.73% vs inactive retention...,KPI card
1,Churn Contribution Risk,Inactive customers contribute more churn than ...,Inactive customers: 48.49% customer share but ...,Churn contribution chart
2,Product Depth Risk,High product load customers show extreme churn...,High Product Load + Inactive churn rate: 90.22%,Product-depth risk chart
3,Best Retention Profile,Active customers with optimal product depth sh...,Optimal Product Depth + Active churn rate: 5.56%,Retention profile insight card
4,Geographic Risk Segment,Germany inactive customers show the highest ge...,Germany + Inactive churn rate: 41.08%,Geography risk chart
5,High-Value Disengagement Risk,High-value disengaged customers show elevated ...,High-value disengaged churn rate: 29.06%,High-value disengaged detector
6,Relationship Strength Signal,Active customers have much stronger relationsh...,Active RSI: 79.07 vs inactive RSI: 48.91,Relationship Strength KPI


### Export Engagement Analysis Outputs

In [20]:
# Create engagement analysis output directory
ENGAGEMENT_OUTPUT_DIR = BASE_DIR / "data" / "processed" / "engagement_analysis"
ENGAGEMENT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Export key engagement analysis outputs
engagement_distribution.to_csv(
    ENGAGEMENT_OUTPUT_DIR / "engagement_distribution.csv",
    index=False
)

engagement_churn_summary.to_csv(
    ENGAGEMENT_OUTPUT_DIR / "engagement_churn_summary.csv",
    index=False
)

engagement_churn_contribution.to_csv(
    ENGAGEMENT_OUTPUT_DIR / "engagement_churn_contribution.csv",
    index=False
)

product_engagement_churn.to_csv(
    ENGAGEMENT_OUTPUT_DIR / "product_engagement_churn.csv",
    index=False
)

balance_engagement_churn.to_csv(
    ENGAGEMENT_OUTPUT_DIR / "balance_engagement_churn.csv",
    index=False
)

geography_engagement_churn.to_csv(
    ENGAGEMENT_OUTPUT_DIR / "geography_engagement_churn.csv",
    index=False
)

high_value_disengaged_analysis.to_csv(
    ENGAGEMENT_OUTPUT_DIR / "high_value_disengaged_analysis.csv",
    index=False
)

relationship_strength_engagement.to_csv(
    ENGAGEMENT_OUTPUT_DIR / "relationship_strength_engagement.csv",
    index=False
)

engagement_tier_matrix.to_csv(
    ENGAGEMENT_OUTPUT_DIR / "engagement_tier_matrix.csv",
    index=False
)

engagement_kpi_summary.to_csv(
    ENGAGEMENT_OUTPUT_DIR / "engagement_kpi_summary.csv",
    index=False
)

engagement_insight_table.to_csv(
    ENGAGEMENT_OUTPUT_DIR / "engagement_insight_table.csv",
    index=False
)

print("Engagement analysis outputs exported successfully.")
print(f"Output Directory: {ENGAGEMENT_OUTPUT_DIR}")
print("Files Exported: 11")

Engagement analysis outputs exported successfully.
Output Directory: C:\Users\Mohit\OneDrive\Documents\Data Analytics\Unified Mentor Projects\Customer Engagement & Product Utilization Analytics for Retention Strategy\data\processed\engagement_analysis
Files Exported: 11


## Notebook 04 Final Conclusion

Notebook 04 completed the engagement-focused churn analysis for the Customer Engagement & Product Utilization Analytics project.

The analysis confirms that customer activity is a strong retention driver. Active customers show a higher retention rate, lower churn rate, and stronger relationship strength compared with inactive customers.

Key final findings:

- Active customers have a retention rate of approximately 85.73%.
- Inactive customers have a retention rate of approximately 73.15%.
- Active customer retention is around 1.17x stronger than inactive customer retention.
- Inactive customers carry approximately 1.88x higher churn risk than active customers.
- Inactive customers represent 48.49% of customers but contribute 63.92% of total churn.
- High Product Load customers show extreme churn risk, especially inactive customers.
- Active customers with optimal product depth show the strongest retention behavior.
- Germany inactive customers form a high-risk geography-engagement segment.
- High-value disengaged customers represent a major retention opportunity.
- Relationship Strength Index is clearly higher among active customers and useful for dashboard scoring.

The exported engagement analysis outputs are now ready to support the Streamlit dashboard module:

`Engagement vs Churn Overview`

The next notebook will focus on product utilization impact analysis.